# Notebook 1 — Text Corpus (Task 1)

This notebook covers the **data layer for Task 1** (text generation). The corpus is a deterministically generated English text of ~12,000 words across 10 topics (nature, science, technology, history, food, travel, education, health, art, sports). Generation code lives in `task1_text_generation/data/generate_dataset.py` — this notebook imports that module and visualizes the output.

> Task 2's QA dataset is in a separate notebook at [`../../task2_chatbot/notebooks/01_data.ipynb`](../../task2_chatbot/notebooks/01_data.ipynb).

## Setup

Add the project root (two levels up from this notebook) to `sys.path` so `task1_text_generation.*` and `shared.*` are importable.

In [ ]:
import os
import sys
from collections import Counter

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), os.pardir, os.pardir))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

DATA_DIR = os.path.join(PROJECT_ROOT, "task1_text_generation", "data")
print(f"Project root: {PROJECT_ROOT}")
print(f"Data dir:     {DATA_DIR}")

## Generating (or loading) the corpus

The generator walks across 10 topics. For each topic it produces multi-paragraph sections using:

- A per-topic vocabulary of subjects, verbs, adjectives, objects, adverbs.
- ~20 sentence templates plus intro/conclusion templates.
- Occasional connector phrases ("Furthermore,", "In contrast,", …) for discourse variety.

This gives us real English surface structure without pulling in any pretrained data.

In [ ]:
from task1_text_generation.data.generate_dataset import generate_corpus

corpus_path = os.path.join(DATA_DIR, "corpus.txt")

if not os.path.exists(corpus_path):
    print("Generating corpus (seed=42)...")
    corpus = generate_corpus(target_words=12000, seed=42)
    with open(corpus_path, "w", encoding="utf-8") as f:
        f.write(corpus)
else:
    with open(corpus_path, "r", encoding="utf-8") as f:
        corpus = f.read()

print(f"Corpus length: {len(corpus):,} characters")
print(f"Word count:    {len(corpus.split()):,} words")
print(f"Unique words:  {len(set(corpus.lower().split())):,}")
print(f"Unique chars:  {len(set(corpus))}")

### Sample of the generated text

Sections are separated by `\n\n---\n\n`. Here is the first section so we can inspect the style:

In [ ]:
first_section = corpus.split("\n\n---\n\n")[0]
print(first_section)

## Corpus statistics

Basic distributions that will matter when we size tokenizers and embeddings in the next notebook.

In [ ]:
import matplotlib.pyplot as plt

words = corpus.lower().split()
word_freq = Counter(words)
word_lengths = [len(w.strip(".,!?;:\"'")) for w in words]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(word_lengths, bins=range(1, 20), edgecolor="black", alpha=0.8)
axes[0].set_title("Word length distribution")
axes[0].set_xlabel("characters per word")
axes[0].set_ylabel("count")

top_words = word_freq.most_common(20)
axes[1].barh([w for w, _ in top_words][::-1], [c for _, c in top_words][::-1])
axes[1].set_title("Top 20 most frequent words")
axes[1].set_xlabel("count")

plt.tight_layout()
plt.show()

In [ ]:
char_freq = Counter(corpus.lower())
top_chars = [(c, n) for c, n in char_freq.most_common(25) if c.strip()]

plt.figure(figsize=(12, 3.5))
plt.bar([c for c, _ in top_chars], [n for _, n in top_chars])
plt.title("Character frequency (top 25 non-whitespace)")
plt.ylabel("count")
plt.show()

## Summary

- ~12k-word corpus written to `task1_text_generation/data/corpus.txt` — reproducible via `seed=42`.
- Next: [`02_text_generation.ipynb`](02_text_generation.ipynb) trains the six text-generation models on this corpus.